# Analysis of the Allegheny County Jail Oversight Board Meeting Minutes

- Contributor: Afifa Iqbal
- AI Acknowledgements: Claude was used
- There were a lot of changes made, iteration over the different stop words and all of these have beeen made on the basis of the LDA model which was done repeatedly

## Topic Modeling Analysis

This notebook contains code that performs Topic Modeling Analysis, using LDA, of pre-processed JOB meeting minutes text. 

### Import libraries and lemmatized text data:

In [32]:
# need to ask Anna to remove the years, some common words such as adjourn, order etc. Have to make a list to give to her

In [548]:
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

# Set up paths, taken from preprocessing notebooks
from pathlib import Path

BASE = Path("..").resolve()
OUT_DIR = BASE / "Data" / "Text" # where lemmatized text is stored

path = OUT_DIR / 'lemmatized_text.xz'

# Adapted from UDA Clustering on Text.ipynb
import pickle

with open(path, 'rb') as f:
    documents = pickle.load(f)



#print(f'Number of documents: {len(documents)}')
#print(f'Document # 1:{documents[0]}')
#print(f'Document # 1:{len(documents[0])}')
#print('---------------------------------------------------------------------------\n')
#print(f'Document # 77:{documents[76]}')
#print(f'Document # 77:{len(documents[76])}')
#print('---------------------------------------------------------------------------\n')
#print(f'Document # 156{documents[155]}')
#print(f'Document # 156:{len(documents[155])}')

f#or i, doc in enumerate(documents):
  #  print(f'Document {i}: {len(doc)}')

print(len(documents))

156


## Cleaning the Lammatized Text 
##### Steps taken:
###### - removing the documents which are empty
###### - removing the digits and numbers (except for years)
###### - removing custom stop words based on the analysis from LDA so far
###### - Creating another document list which contains under 10k words, in case that helps with the situation of repeatetive words

In [549]:
print([i for i, doc in enumerate(documents) if len(doc) == 0]) #removing the empty docs as well

[34, 43, 44]


In [550]:
filtered_docs = []
filtered_names = []

for i, doc in enumerate(documents):
    if len(doc) > 0:  # only keep non-empty documents
        filtered_docs.append(doc)
        filtered_names.append(doc_names[i])
        
print (len(filtered_docs))

153


In [551]:
cleaned_docs = []
for doc in filtered_docs:
    cleaned_tokens = [
        word for word in doc
        if not word.isdigit()  # removing numbers as they were showing up as the frequent words in the vocab list
        or (word.isdigit() and len(word) == 4 and word.startswith(('19', '20')))] # keeping years to help with policy questions related to time
    cleaned_docs.append(cleaned_tokens)

In [552]:
print(len(cleaned_docs))

153


In [553]:
print(type(cleaned_docs))

<class 'list'>


In [554]:
# creating a stop words list by reviewing the most frequent words which are not meaningful 
# these words were observed in the initial LDA model 

from collections import Counter
all_words = [word for doc in cleaned_docs for word in doc]
word_counts = Counter(all_words)
print(word_counts.most_common(300))

[('judge', 11507), ('jail', 11221), ('hallam', 10109), ('ms', 9973), ('warden', 7770), ('know', 7386), ('board', 6141), ('like', 5935), ('people', 5698), ('county', 5013), ('think', 4938), ('time', 4795), ('deputy', 4438), ('want', 4408), ('okay', 4366), ('thank', 4303), ('report', 4114), ('meeting', 4085), ('work', 4042), ('individual', 3961), ('question', 3915), ('e', 3912), ('ms.', 3748), ('mr', 3606), ('evashavik', 3457), ('t', 3290), ('clark', 3277), ('come', 3272), ('.', 3172), ('thing', 3136), ('need', 2931), ('right', 2906), ('s', 2897), ('ask', 2831), ('staff', 2770), ('month', 2726), ('provide', 2574), ('health', 2548), ('comment', 2513), ('inmate', 2487), ('talk', 2473), ('use', 2451), ('program', 2449), ('howsie', 2379), ('day', 2366), ('member', 2313), ('allegheny', 2191), ('harper', 2191), ('look', 2157), ('issue', 2137), ('public', 2091), ('medical', 2080), ('facility', 2062), ('year', 1994), ('yes', 1991), ('chief', 1965), ('acj', 1909), ('number', 1861), ('good', 1852)

In [555]:
custom_stop_words = [
    # fillers during conversations - should not be meaningful
    'know', 'think', 'want', 'okay', 'thank',
    'actually', 'thing', 'ask', 'look', 'say', 'yes', 'yeah', 'sure',
    'oh', 'sorry', 'maybe', 'lot', 
    'let', 'bring', 'tell', 'come', 'happen', 'hear',
    'speak', 'try', 'find', 'understand', 'able',
    'way', 'today', 'anybody', 'everybody', 'somebody', 'guy', 'folk',
    'far', 'past', 'long', 'different', 'specific', 'specifically',
    'currently', 'actually', 'forward', 'outside', 'available', 'little',
    'couple', 'ago', 'appreciate', 'mention', 'run', 'add', 'end', 'page'
    
    # some of the single letters are showing up as frequent words
    'e', 't', 's', 'o', 'n', 'r', 'h', 'w', 'c', 'd', 'l', 'g',
    'f', 'p', 'u', 'm', 'y', "'"
    
    # meeting formal words
    'meeting', 'motion', 'second', 'approve', 'adjourn', 'report',
    'member', 'present', 'minute', 'vote', 'board', 'committee',
    'comment', 'public', 'business', 'follow', 'address', 'discussion',
    'record', 'hold', 'begin', 'complete', 'submit', 'review', 'executive', 'president',
    
    # places/locations
    'allegheny', 'county', 'pennsylvania', 'pittsburgh',
    
    # names of specific board numbers maybe?
    'hallam', 'evashavik', 'howsie', 'beasom', 'lazzara', 'brinkman',
    'toma', 'bigley', 'innamorato', 'damick', 'wagner', 'moss',
    'klein', 'perkins', 'williams', 'clark', 'bethany', 'ludwig', 'mcdaniel', 'ron', 'dana', 'chuck', 'doug',
    'larry', 'pofi', 'acchs', 'martoni', 'madarino', 'catanese',
    'phillips', 'donna', 'jo', 'joe', 'orlando', 'harper', 'latoya',
    'mullen', 'fitzgerald', 'asturi', 'carol', 'hertz', 'mike',
    'gilmore', 'marion', 'chelsa', 'william', 'dilucente', 'cashman',
    'walker', 'corizon', 'wingard', 'trevor', 'defazio', 'korinski'
]

In [556]:
print (ENGLISH_STOP_WORDS)

frozenset({'while', 'from', 'me', 'seeming', 'less', 'and', 'became', 'formerly', 'six', 'beforehand', 'their', 'had', 'one', 'keep', 'whence', 'yours', 'but', 'front', 'same', 'some', 'his', 'onto', 'because', 'who', 'without', 'perhaps', 'ever', 'mine', 'thereby', 'i', 'nowhere', 'therein', 'give', 'everywhere', 'we', 'by', 'bottom', 'being', 'which', 'any', 'than', 'through', 'even', 'why', 'over', 'out', 'few', 'nevertheless', 'not', 'thence', 'interest', 'next', 'de', 'whereupon', 'would', 'within', 'much', 'many', 'amount', 'hasnt', 'or', 'meanwhile', 'move', 'together', 'always', 'those', 'too', 'whom', 'himself', 'were', 'becomes', 'often', 'under', 'least', 'please', 'only', 'fifteen', 'twelve', 'cry', 'for', 'are', 'back', 'toward', 'thin', 'us', 'thus', 'throughout', 'none', 'whole', 'almost', 'never', 'was', 'must', 'two', 'per', 'before', 'made', 'whether', 'own', 'four', 'somehow', 'empty', 'part', 'these', 'towards', 'whose', 'eight', 'someone', 'been', 'mill', 'anyway',

In [557]:
#from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
#all_stop_words = list(ENGLISH_STOP_WORDS) + custom_stops

In [558]:
#print (all_stop_words)

In [559]:
#print (len(filtered_docs))

In [574]:
## based on the LDA Analysis that was done before

recurring_commenter_words = [
    'condom', 'stabilization', 'exorbitant', 'clowney', 
    'denial', 'intentionally', 'firsthand', 'unanswered',
    'revisit', 'indication', 'challenging', 'destroy',
    'hell', 'judgment', 'behave', 'comprise', 'desperately',
    'opposition', 'audio', 'owner', 'technological', 'wow',
    'traumatic', 'blame', 'horrible', 'frustration', 'rescind'
]
custom_stop_words.extend(recurring_commenter_words)

In [575]:
# Source - https://stackoverflow.com/a/46328592
# Posted by PyRsquared
# Retrieved 2026-04-22, License - CC BY-SA 3.0

stop_words_removed_doc = [
    [word for word in doc if word not in custom_stop_words]
    for doc in cleaned_docs]

In [576]:
#print(f'Number of documents: {len(stop_words_removed_doc)}')
#print(f'Document # 1:{len(stop_words_removed_doc[0])}')
#print('---------------------------------------------------------------------------\n')
#print(f'Document # 1:{len(stop_words_removed_doc[76])}')
#print('---------------------------------------------------------------------------\n')
#print(f'Document # 1:{len(stop_words_removed_doc[155])}')


# this step is being done to understand how many words are being lost due to the custom stop words
# and to figure out if there are unreasonable long docs 
for i, (doc1, doc2) in enumerate(zip(filtered_docs, stop_words_removed_doc)):
    print(f"Document {i+1}: Original = {len(doc1)} words | Cleaned = {len(doc2)} words")
    if len(doc2) > 10000:
        print(f"--------------------------- Document {i+1}: Cleaned = {len(doc2)} words ------------------------------")

# it was noticed that some of the documents had a large number of words, as compared to the rest
# to ensure that it is not creating the issue in the model, another test lammatized text is being created to check the LDA Model

docs_10k = []
docs_10k_names = []
for i, doc in enumerate(stop_words_removed_doc):
    if len(doc) < 10000:  # only keep less than 10k words documents
        #print(f"--------------------------- Document {i+1}: Cleaned = {len(doc)} words ------------------------------")
        docs_10k.append(doc)
        docs_10k_names.append(doc_names[i])
        
print (f'\n Number of Documents Containing Less than 10K words:{len(docs_10k)}')

Document 1: Original = 350 words | Cleaned = 230 words
Document 2: Original = 493 words | Cleaned = 342 words
Document 3: Original = 485 words | Cleaned = 326 words
Document 4: Original = 545 words | Cleaned = 415 words
Document 5: Original = 261 words | Cleaned = 166 words
Document 6: Original = 694 words | Cleaned = 530 words
Document 7: Original = 499 words | Cleaned = 342 words
Document 8: Original = 763 words | Cleaned = 562 words
Document 9: Original = 506 words | Cleaned = 324 words
Document 10: Original = 679 words | Cleaned = 482 words
Document 11: Original = 357 words | Cleaned = 241 words
Document 12: Original = 377 words | Cleaned = 253 words
Document 13: Original = 642 words | Cleaned = 483 words
Document 14: Original = 506 words | Cleaned = 372 words
Document 15: Original = 846 words | Cleaned = 606 words
Document 16: Original = 590 words | Cleaned = 436 words
Document 17: Original = 787 words | Cleaned = 632 words
Document 18: Original = 727 words | Cleaned = 531 words
D

In [577]:
#print(type(stop_words_removed_doc))

In [578]:
print(stop_words_removed_doc[0][:70])

['monthly', 'meeting', 'jail', 'oversight', 'thursday', 'october', '2012', 'conference', 'room', 'court', 'house', '4:00', 'p.m.', 'following', 'honorable', 'judge', 'kimberly', 'm.', 'gayle', 'represent', 'honorable', 'elliott', 'chief', 'defender', 'austin', 'davis', 'represent', 'honorable', 'rich', 'represent', 'honorable', 'dr.', 'charles', 'honorable', 'sheriff', 'warden', 'major', 'interested', 'party', 'mrs.', 'discuss', 'jail', 'request', 'following', 'discuss', 'role', 'warden', 'like', 'warden', 'concentrate', 'library', 'learning', 'teaching', 'inmate', 'read', 'provide', 'warden', 'brochure', 'pa', 'prison', 'society', 'inform', 'warden', 'supreme', 'court', 'rule', 'state', 'pay', 'jail', 'september']


In [579]:
#joining the tokenized text into a string for tf-idf

docs_string = [' '.join(doc) for doc in stop_words_removed_doc]
#print(docs_string[0])

docs_string2 = [' '.join(doc) for doc in docs_10k]

### Creating TF-IDF tables to analyze the most frequently used words

##### The following code is used from UDA Topic Modeling code Demo. 

In [580]:
#vocab_size = 10000
from sklearn.feature_extraction.text import TfidfVectorizer

#upon analyzing further, it was found many of the numbers are getting added to the library which are meaningless for our analysis,
#so toekn_parameter was added for it to be only alphabest and at least length of 3

tfidf_vectorizer1 = TfidfVectorizer(max_df=0.7, min_df=15, token_pattern=r'[a-zA-Z]{3,}', max_features=10000)
tfidf1 = tfidf_vectorizer1.fit_transform(docs_string)

tfidf_vectorizer2 = TfidfVectorizer(max_df=0.9,min_df=10,token_pattern=r'[a-zA-Z]{3,}', max_features=5000)
tfidf2 = tfidf_vectorizer2.fit_transform(docs_string)

tfidf_vectorizer3 = TfidfVectorizer(max_df=0.9, min_df=10, token_pattern=r'[a-zA-Z]{3,}', max_features=10000)
tfidf3 = tfidf_vectorizer3.fit_transform(docs_string)

tfidf_vectorizer4 = TfidfVectorizer(max_df=0.9, min_df=15,token_pattern=r'[a-zA-Z]{3,}', max_features=5000)
tfidf4 = tfidf_vectorizer4.fit_transform(docs_string)

tfidf_vectorizer5 = TfidfVectorizer(max_df=0.8, min_df=10, token_pattern=r'[a-zA-Z]{3,}', max_features=7000)
tfidf5 = tfidf_vectorizer5.fit_transform(docs_string)

#----------------------------- Documents having under 10K words Models ------------------------------

tfidf_vectorizer6 = TfidfVectorizer(max_df=0.9, min_df=10, token_pattern=r'[a-zA-Z]{3,}', max_features=5000)
tfidf6 = tfidf_vectorizer6.fit_transform(docs_string2)

tfidf_vectorizer7 = TfidfVectorizer(max_df=0.8, min_df=15, token_pattern=r'[a-zA-Z]{3,}', max_features=7000)
tfidf7 = tfidf_vectorizer7.fit_transform(docs_string2)

tfidf_vectorizer8 = TfidfVectorizer(max_df=0.7, min_df=15, token_pattern=r'[a-zA-Z]{3,}', max_features=10000)
tfidf8 = tfidf_vectorizer8.fit_transform(docs_string2)

tfidf_vectorizer9 = TfidfVectorizer(max_df=0.8, min_df=10, token_pattern=r'[a-zA-Z]{3,}', max_features=10000)
tfidf9 = tfidf_vectorizer9.fit_transform(docs_string2)

tfidf_vectorizer10 = TfidfVectorizer(max_df=0.9, min_df=10, token_pattern=r'[a-zA-Z]{3,}', max_features=10000)
tfidf10 = tfidf_vectorizer10.fit_transform(docs_string2)

# making a list of all to create the loops later on
tfidf_matrices = [tfidf1, tfidf2, tfidf3, tfidf4, tfidf5, 
                  tfidf6, tfidf7, tfidf8, tfidf9, tfidf10]

vectorizers = [tfidf_vectorizer1, tfidf_vectorizer2, tfidf_vectorizer3, tfidf_vectorizer4, tfidf_vectorizer5,
               tfidf_vectorizer6, tfidf_vectorizer7, tfidf_vectorizer8, tfidf_vectorizer9, tfidf_vectorizer10]


In [581]:

# Extract vocabs and print length + tfidf shape
vocabs = []
for idx, (vectorizer, tfidf_matrix) in enumerate(zip(vectorizers, tfidf_matrices)):
    vocab = vectorizer.get_feature_names_out()
    vocabs.append(vocab)
    print(f"TF-IDF {idx+1}: vocab={len(vocab)}, shape={tfidf_matrix.shape}")

TF-IDF 1: vocab=2293, shape=(153, 2293)
TF-IDF 2: vocab=3070, shape=(153, 3070)
TF-IDF 3: vocab=3070, shape=(153, 3070)
TF-IDF 4: vocab=2379, shape=(153, 2379)
TF-IDF 5: vocab=3033, shape=(153, 3033)
TF-IDF 6: vocab=2675, shape=(141, 2675)
TF-IDF 7: vocab=2031, shape=(141, 2031)
TF-IDF 8: vocab=1989, shape=(141, 1989)
TF-IDF 9: vocab=2640, shape=(141, 2640)
TF-IDF 10: vocab=2675, shape=(141, 2675)


### Fitting LDA Model to the Data

##### The following code is used from UDA Topic Modeling code Demo. 

In [582]:


from sklearn.decomposition import LatentDirichletAllocation

lda_configs = [
    {'n_components': 10, 'max_iter': 30, 'random_state': 0},
    {'n_components': 8,  'max_iter': 50, 'doc_topic_prior': 0.1, 'topic_word_prior': 0.01, 
     'learning_method': 'online', 'learning_offset': 50.0, 'random_state': 0},
    {'n_components': 8,  'max_iter': 50, 'learning_offset': 50.0, 'random_state': 0},
]

lda_models = []
topic_word_dists = []
model_labels = []

for tfidf_idx, tfidf_matrix in enumerate(tfidf_matrices):
    for lda_config in lda_configs:
        lda = LatentDirichletAllocation(**lda_config)
        lda.fit(tfidf_matrix)
        
        dist = np.array([row / row.sum() for row in lda.components_])
        
        lda_models.append(lda)
        topic_word_dists.append(dist)
        label = f"TFIDF-{tfidf_idx+1}_LDA-{lda_config['n_components']}topics_iter{lda_config['max_iter']}"
        model_labels.append(label)
        
        print(f"{label}: components shape={lda.components_.shape}, dist shape={dist.shape}")

TFIDF-1_LDA-10topics_iter30: components shape=(10, 2293), dist shape=(10, 2293)
TFIDF-1_LDA-8topics_iter50: components shape=(8, 2293), dist shape=(8, 2293)
TFIDF-1_LDA-8topics_iter50: components shape=(8, 2293), dist shape=(8, 2293)
TFIDF-2_LDA-10topics_iter30: components shape=(10, 3070), dist shape=(10, 3070)
TFIDF-2_LDA-8topics_iter50: components shape=(8, 3070), dist shape=(8, 3070)
TFIDF-2_LDA-8topics_iter50: components shape=(8, 3070), dist shape=(8, 3070)
TFIDF-3_LDA-10topics_iter30: components shape=(10, 3070), dist shape=(10, 3070)
TFIDF-3_LDA-8topics_iter50: components shape=(8, 3070), dist shape=(8, 3070)
TFIDF-3_LDA-8topics_iter50: components shape=(8, 3070), dist shape=(8, 3070)
TFIDF-4_LDA-10topics_iter30: components shape=(10, 2379), dist shape=(10, 2379)
TFIDF-4_LDA-8topics_iter50: components shape=(8, 2379), dist shape=(8, 2379)
TFIDF-4_LDA-8topics_iter50: components shape=(8, 2379), dist shape=(8, 2379)
TFIDF-5_LDA-10topics_iter30: components shape=(10, 3033), dist s

In [583]:
# Step 4 - Print topics for all 30 models
def print_top_words(tf_1d_table, vocab, num_top_words=15):
    sorted_tuples = sorted(zip(tf_1d_table, vocab), reverse=True)[:num_top_words]
    for _, word in sorted_tuples:
        print(word, end=' ')
    print()

for model_idx, (dist, label) in enumerate(zip(topic_word_dists, model_labels)):
    tfidf_idx = model_idx // 3   # which tfidf this model used
    vocab = vocabs[tfidf_idx]    # get matching vocab
    
    print(f"\n{'='*60}")
    print(f"Model: {label}")
    print(f"{'='*60}")
    
    for topic_idx in range(dist.shape[0]):
        print(f"[Topic {topic_idx}]")
        print_top_words(dist[topic_idx], vocab)


Model: TFIDF-1_LDA-10topics_iter30
[Topic 0]
absolute removal laugh secret ring frequency foremost xylazine brand disrespect wheel contrary terrance intervene formally 
[Topic 1]
absolute removal laugh secret ring frequency foremost xylazine brand disrespect wheel contrary terrance intervene formally 
[Topic 2]
absolute removal laugh secret ring frequency foremost xylazine brand disrespect wheel contrary terrance intervene formally 
[Topic 3]
absolute removal laugh secret ring frequency foremost xylazine brand disrespect wheel contrary terrance intervene formally 
[Topic 4]
absolute removal laugh secret ring frequency foremost xylazine brand disrespect wheel contrary terrance intervene formally 
[Topic 5]
dilucente right talk person chief honorable incarcerated davis tablet mean medication book man answer connor 
[Topic 6]
absolute removal laugh secret ring frequency foremost xylazine brand disrespect wheel contrary terrance intervene formally 
[Topic 7]
absolute removal laugh secret 

In [571]:
problem_phrases = ['condom', 'stabilization', 'exorbitant', 'clowney']
for phrase in problem_phrases:
    docs = [(i, filtered_names[i]) for i, doc in enumerate(stop_words_removed_doc) if phrase in doc]
    print(f"'{phrase}': {docs}")

'condom': [(100, '2022_10_0_JOB_Minutes.txt'), (113, '2023_11_0_JOB_Minutes.txt'), (114, '2023_12_0_JOB_Minutes.txt'), (124, '2024_10_0_JOB_Minutes.txt'), (125, '2024_11_0_JOB_Minutes.txt'), (126, '2024_12_0_JOB_Minutes.txt'), (127, '2024_1_0_JOB_Minutes.txt'), (128, '2024_2_0_JOB_Minutes.txt'), (130, '2024_3_0_JOB_Minutes.txt'), (132, '2024_4_0_JOB_Minutes.txt'), (133, '2024_5_0_JOB_Minutes.txt'), (134, '2024_6_0_JOB_Minutes.txt'), (135, '2024_7_0_JOB_Minutes.txt'), (136, '2024_8_0_JOB_Minutes.txt'), (137, '2024_9_0_JOB_Minutes.txt')]
'stabilization': [(101, '2022_11_0_JOB_Minutes.txt'), (102, '2022_12_0_JOB_Minutes.txt'), (112, '2023_10_0_JOB_Minutes.txt'), (113, '2023_11_0_JOB_Minutes.txt'), (114, '2023_12_0_JOB_Minutes.txt'), (115, '2023_1_0_JOB_Minutes.txt'), (116, '2023_2_0_JOB_Minutes.txt'), (117, '2023_3_0_JOB_Minutes.txt'), (118, '2023_4_0_JOB_Minutes.txt'), (119, '2023_5_0_JOB_Minutes.txt'), (121, '2023_7_0_JOB_Minutes.txt'), (122, '2023_8_0_JOB_Minutes.txt'), (123, '2023_9_0